# LSTM Model Training Notebook

Train the dynamic hand-sign LSTM model using preprocessed sequence data and save artifacts used by the backend.

Expected files:
- data/X_data.npy
- data/y_data.npy
- models/wlasl_labels.npy (optional, for readable class names)

## 1) Install and Import Dependencies

In [8]:
# Uncomment if TensorFlow is missing in your current kernel environment.
# %pip install -q tensorflow numpy scikit-learn matplotlib seaborn

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

print('TensorFlow version:', tf.__version__)
print('Timestamp:', datetime.now().isoformat(timespec='seconds'))

TensorFlow version: 2.19.0
Timestamp: 2026-03-17T22:54:18


In [9]:
# Cell 4: Bootstrap dataset availability for remote kernels (optional).
# Run this only when Cell 5 cannot find data files.

import shutil
from pathlib import Path

data_dir = Path('data')
models_dir = Path('models')
data_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

x_dst = data_dir / 'X_data.npy'
y_dst = data_dir / 'y_data.npy'
labels_dst = models_dir / 'wlasl_labels.npy'

def find_repo_roots_with_arrays(search_roots):
    matches = []
    seen = set()
    for base in search_roots:
        if not base.exists():
            continue
        try:
            for data_dir_candidate in base.rglob('data'):
                if not data_dir_candidate.is_dir():
                    continue
                repo_root = data_dir_candidate.parent
                x_src = repo_root / 'data' / 'X_data.npy'
                y_src = repo_root / 'data' / 'y_data.npy'
                if x_src.exists() and y_src.exists():
                    key = str(repo_root)
                    if key not in seen:
                        seen.add(key)
                        matches.append(repo_root)
                        if len(matches) >= 20:
                            return matches
        except (OSError, PermissionError):
            continue
    return matches

if x_dst.exists() and y_dst.exists():
    print('Dataset already present in current kernel filesystem at ./data')
else:
    search_roots = [
        Path('/content'),
        Path('/content/drive/MyDrive'),
        Path('/mnt'),
        Path('/workspace'),
        Path('/workspaces'),
        Path('/app'),
    ]
    candidates = find_repo_roots_with_arrays(search_roots)

    if candidates:
        selected = candidates[0]
        shutil.copy2(selected / 'data' / 'X_data.npy', x_dst)
        shutil.copy2(selected / 'data' / 'y_data.npy', y_dst)
        labels_src = selected / 'models' / 'wlasl_labels.npy'
        if labels_src.exists():
            shutil.copy2(labels_src, labels_dst)
        print('Copied dataset from:', selected)
        if len(candidates) > 1:
            print('Note: multiple dataset candidates were found; using the first match.')
            for idx, cand in enumerate(candidates[:5], start=1):
                print(f'  {idx}. {cand}')
    else:
        print('No accessible dataset found in this kernel environment.')
        print('Place files in this runtime: data/X_data.npy and data/y_data.npy')
        print('Optional label file: models/wlasl_labels.npy')

print('X exists:', x_dst.exists())
print('y exists:', y_dst.exists())
print('labels exists:', labels_dst.exists())

No accessible dataset found in this kernel environment.
Place files in this runtime: data/X_data.npy and data/y_data.npy
Optional label file: models/wlasl_labels.npy
X exists: False
y exists: False
labels exists: False


In [12]:
# Cell 5: Manual or auto source-path copy helper.
# Use this when Cell 4 cannot find data automatically.
from pathlib import Path
import os
import shutil

data_dir = Path('data')
models_dir = Path('models')
data_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

x_dst = data_dir / 'X_data.npy'
y_dst = data_dir / 'y_data.npy'
labels_dst = models_dir / 'wlasl_labels.npy'

# Optional explicit source path. Leave as None to auto-discover in this kernel filesystem.
SOURCE_ROOT = None  # Example: Path('/content/your_folder/hand_sign_detection_dynamic')

def is_valid_repo_root(root: Path) -> bool:
    return (root / 'data' / 'X_data.npy').exists() and (root / 'data' / 'y_data.npy').exists()

def find_candidate_roots_rglob():
    search_roots = [
        Path('/content'),
        Path('/tmp'),
        Path('/mnt'),
        Path('/workspace'),
        Path('/workspaces'),
        Path('/app'),
        Path.cwd(),
    ]
    matches = []
    seen = set()

    for base in search_roots:
        if not base.exists():
            continue
        try:
            for x_file in base.rglob('X_data.npy'):
                if x_file.parent.name != 'data':
                    continue
                candidate = x_file.parent.parent
                key = str(candidate)
                if key in seen:
                    continue
                if is_valid_repo_root(candidate):
                    seen.add(key)
                    matches.append(candidate)
                    if len(matches) >= 10:
                        return matches
        except (OSError, PermissionError):
            continue
    return matches

def find_candidate_roots_walk(max_dirs: int = 8000):
    # Bounded filesystem walk for non-standard mounts (for example /tmp-mounted workspaces).
    skip_names = {
        'proc', 'sys', 'dev', 'run', 'var', 'snap', 'nix', 'lost+found',
    }
    roots = ['/','/tmp','/home','/root','/mnt','/content']
    matches = []
    seen = set()
    visited = 0

    for start in roots:
        if not Path(start).exists():
            continue
        for dirpath, dirnames, _ in os.walk(start):
            visited += 1
            if visited > max_dirs:
                return matches

            # Prune expensive/system directories.
            dirnames[:] = [
                d for d in dirnames
                if d not in skip_names and not d.startswith('.')
            ]

            root = Path(dirpath)
            if is_valid_repo_root(root):
                key = str(root)
                if key not in seen:
                    seen.add(key)
                    matches.append(root)
                    if len(matches) >= 10:
                        return matches
    return matches

selected_root = None
if SOURCE_ROOT is not None:
    selected_root = Path(SOURCE_ROOT)
    if not is_valid_repo_root(selected_root):
        raise FileNotFoundError(
            f'SOURCE_ROOT is invalid for this kernel: {selected_root}. '
            'Expected data/X_data.npy and data/y_data.npy.'
        )
else:
    candidates = find_candidate_roots_rglob()
    if not candidates:
        candidates = find_candidate_roots_walk()
    if candidates:
        selected_root = candidates[0]
        print('Auto-discovered dataset root:', selected_root)
        if len(candidates) > 1:
            print('Additional candidates:')
            for idx, cand in enumerate(candidates[1:5], start=2):
                print(f'  {idx}. {cand}')

if selected_root is None:
    raise FileNotFoundError(
        'No dataset files are visible to this kernel after deep scan. '
        'Upload X_data.npy and y_data.npy into this runtime (for example under /content), '
        'or set SOURCE_ROOT to a visible dataset root and rerun Cell 5.'
    )

shutil.copy2(selected_root / 'data' / 'X_data.npy', x_dst)
shutil.copy2(selected_root / 'data' / 'y_data.npy', y_dst)
labels_src = selected_root / 'models' / 'wlasl_labels.npy'
if labels_src.exists():
    shutil.copy2(labels_src, labels_dst)

print('Copied from:', selected_root)
print('X exists:', x_dst.exists(), 'size:', x_dst.stat().st_size if x_dst.exists() else 0)
print('y exists:', y_dst.exists(), 'size:', y_dst.stat().st_size if y_dst.exists() else 0)
print('labels exists:', labels_dst.exists())

FileNotFoundError: No dataset files are visible to this kernel after deep scan. Upload X_data.npy and y_data.npy into this runtime (for example under /content), or set SOURCE_ROOT to a visible dataset root and rerun Cell 5.

In [ ]:
# Cell 6: Upload dataset files into this runtime (use when Drive/root scan finds nothing).
from pathlib import Path
import shutil

data_dir = Path('data')
models_dir = Path('models')
data_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files  # type: ignore
except Exception as exc:
    raise RuntimeError(
        'Upload helper is only available in Colab kernels. '
        'Use SOURCE_ROOT in Cell 5 or copy files manually into ./data.'
    ) from exc

print('Select at least X_data.npy and y_data.npy. Optionally include wlasl_labels.npy.')
uploaded = files.upload()

for fname in uploaded.keys():
    src = Path(fname)
    lower = fname.lower()
    if lower.endswith('x_data.npy'):
        shutil.move(str(src), str(data_dir / 'X_data.npy'))
    elif lower.endswith('y_data.npy'):
        shutil.move(str(src), str(data_dir / 'y_data.npy'))
    elif lower.endswith('wlasl_labels.npy'):
        shutil.move(str(src), str(models_dir / 'wlasl_labels.npy'))

print('X exists:', (data_dir / 'X_data.npy').exists())
print('y exists:', (data_dir / 'y_data.npy').exists())
print('labels exists:', (models_dir / 'wlasl_labels.npy').exists())

In [ ]:
import os
from pathlib import Path

cwd = Path.cwd()
required_files = [
    Path('data') / 'X_data.npy',
    Path('data') / 'y_data.npy',
]

def has_required_files(root: Path) -> bool:
    return all((root / rel_path).exists() for rel_path in required_files)

def add_candidate(candidate, candidates, seen):
    if not candidate:
        return
    try:
        candidate_path = Path(candidate).expanduser()
    except (TypeError, ValueError):
        return

    key = str(candidate_path)
    if key not in seen:
        seen.add(key)
        candidates.append(candidate_path)

def discover_root_from_scan(scan_roots):
    for scan_root in scan_roots:
        if not scan_root.exists():
            continue

        if has_required_files(scan_root):
            return scan_root

        try:
            for data_dir in scan_root.rglob('data'):
                if data_dir.is_dir():
                    discovered_root = data_dir.parent
                    if has_required_files(discovered_root):
                        return discovered_root
        except (OSError, PermissionError):
            continue

    return None

def discover_colab_drive_repo(base_dir: Path):
    if not base_dir.exists():
        return None
    try:
        for direct_child in base_dir.iterdir():
            if direct_child.is_dir() and has_required_files(direct_child):
                return direct_child

        for data_file in base_dir.rglob('X_data.npy'):
            if data_file.parent.name == 'data':
                candidate_root = data_file.parent.parent
                if has_required_files(candidate_root):
                    return candidate_root
    except (OSError, PermissionError):
        return None
    return None

def try_mount_colab_drive():
    if cwd.as_posix().startswith('/content') and not Path('/content/drive').exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive')
        except Exception:
            pass

try_mount_colab_drive()

candidates = []
seen = set()

# 1) Explicit override from environment.
env_root = os.getenv('HAND_SIGN_REPO_ROOT')
if env_root:
    add_candidate(env_root, candidates, seen)

# 2) Walk up from current working directory.
for candidate in [cwd, *cwd.parents]:
    add_candidate(candidate, candidates, seen)

# 3) Common local/container/WSL/Colab mount paths.
for candidate in [
    Path.home() / 'Documents' / 'hand_sign_detection_dynamic',
    Path('c:/Users/suman/Documents/hand_sign_detection_dynamic'),
    Path('/mnt/c/Users/suman/Documents/hand_sign_detection_dynamic'),
    Path('/app'),
    Path('/workspace/hand_sign_detection_dynamic'),
    Path('/workspaces/hand_sign_detection_dynamic'),
    Path('/content/hand_sign_detection_dynamic'),
    Path('/content/drive/MyDrive/hand_sign_detection_dynamic'),
    Path('/content/drive/MyDrive/Colab Notebooks/hand_sign_detection_dynamic'),
]:
    add_candidate(candidate, candidates, seen)

project_root = next((candidate for candidate in candidates if has_required_files(candidate)), None)

scan_roots = [
    Path('/content'),
    Path('/tmp'),
    Path('/root'),
    Path('/home'),
    Path('/workspace'),
    Path('/workspaces'),
    Path('/mnt'),
]

if project_root is None:
    colab_drive_root = discover_colab_drive_repo(Path('/content/drive/MyDrive'))
    if colab_drive_root is not None:
        project_root = colab_drive_root

if project_root is None:
    project_root = discover_root_from_scan(scan_roots)

if project_root is None:
    attempted_roots = '\n'.join(f' - {candidate}' for candidate in candidates)
    scanned_roots = '\n'.join(f' - {root}' for root in scan_roots if root.exists())
    raise FileNotFoundError(
        'Could not locate required training arrays. Expected both data/X_data.npy and data/y_data.npy.\n'
        f'Current cwd: {cwd}\n'
        f'Tried roots:\n{attempted_roots}\n'
        f'Scanned base roots:\n{scanned_roots}\n'
        'Data files are not visible to this kernel. Run Cell 4, then Cell 5, then Cell 6 upload if needed, and rerun Cell 7.'
    )

x_path = project_root / 'data' / 'X_data.npy'
y_path = project_root / 'data' / 'y_data.npy'
labels_path = project_root / 'models' / 'wlasl_labels.npy'

X = np.load(x_path)
y = np.load(y_path)
class_names = np.load(labels_path, allow_pickle=True) if labels_path.exists() else None

print('Resolved project_root:', project_root)
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Labels available:', class_names is not None)
if class_names is not None:
    print('Number of class names:', len(class_names))

Mounted at /content/drive


FileNotFoundError: Could not locate required training arrays. Expected both data/X_data.npy and data/y_data.npy.
Current cwd: /content
Tried roots:
 - /content
 - /
 - /root/Documents/hand_sign_detection_dynamic
 - c:/Users/suman/Documents/hand_sign_detection_dynamic
 - /mnt/c/Users/suman/Documents/hand_sign_detection_dynamic
 - /app
 - /workspace/hand_sign_detection_dynamic
 - /workspaces/hand_sign_detection_dynamic
 - /content/hand_sign_detection_dynamic
 - /content/drive/MyDrive/hand_sign_detection_dynamic
 - /content/drive/MyDrive/Colab Notebooks/hand_sign_detection_dynamic
Scanned base roots:
 - /content
 - /tmp
 - /root
 - /home
 - /mnt
Data files are not visible to this kernel. Run Cell 4, or Cell 5 with a valid SOURCE_ROOT, then run Cell 6 again.

## 2) Load Dataset

If your notebook kernel starts outside the repository, run Cell 4 first.
If Cell 4 cannot find data, run Cell 5.
If Cell 5 still reports no dataset visible, run Cell 6 to upload X_data.npy and y_data.npy directly into this runtime.
Then run Cell 7 (dataset loader).

Examples of valid SOURCE_ROOT values in remote kernels:
- /content/your_project_root
- /workspace/your_project_root
- /app

In [ ]:
print('Sequence length:', X.shape[1])
print('Feature dimension:', X.shape[2])
print('Total samples:', X.shape[0])

unique_classes, counts = np.unique(y, return_counts=True)
print('Number of unique classes:', len(unique_classes))
print('Minimum class count:', int(counts.min()))
print('Maximum class count:', int(counts.max()))

plt.figure(figsize=(10, 4))
plt.bar(unique_classes[:40], counts[:40])
plt.title('Class Distribution (first 40 classes)')
plt.xlabel('Class index')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 5) Split Data into Train and Validation Sets

In [ ]:
if 'y_cat' not in globals() or 'y_encoded' not in globals():
    class_ids, y_encoded = np.unique(y, return_inverse=True)
    y_encoded = y_encoded.astype(np.int32)
    num_classes = int(len(class_ids))
    y_cat = to_categorical(y_encoded, num_classes=num_classes)

use_stratify = np.bincount(y_encoded).min() >= 2
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded if use_stratify else None,
)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)

## 4) Preprocess Features and Labels

In [ ]:
class_ids, y_encoded = np.unique(y, return_inverse=True)
y_encoded = y_encoded.astype(np.int32)
num_classes = int(len(class_ids))
y_cat = to_categorical(y_encoded, num_classes=num_classes)

print('X dtype:', X.dtype)
print('y categorical shape:', y_cat.shape)
print('num_classes:', num_classes)
print('original class ids:', class_ids.tolist())

## 6) Define Model Architecture

## 8) Train the Model

## 10) Save Trained Model and Artifacts

In [ ]:
required = [
    'project_root', 'model', 'history', 'x_path', 'y_path',
    'X', 'num_classes', 'val_loss', 'val_acc'
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing required objects before saving: {missing}')

if 'target_names' not in globals():
    if class_names is not None and len(class_names) == num_classes:
        target_names = [str(x) for x in class_names]
    else:
        target_names = [f'class_{i}' for i in range(num_classes)]

models_dir = project_root / 'models'
reports_dir = project_root / 'reports'
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

lstm_model_path = models_dir / 'gesture_model.h5'
labels_out_path = models_dir / 'wlasl_labels.npy'
metadata_path = reports_dir / f"lstm_training_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

model.save(lstm_model_path)

if class_names is not None and len(class_names) == num_classes:
    np.save(labels_out_path, class_names)
else:
    np.save(labels_out_path, np.array(target_names))

history_dict = {k: [float(vv) for vv in v] for k, v in history.history.items()}

metadata = {
    'created_at': datetime.now().isoformat(),
    'x_data_path': str(x_path),
    'y_data_path': str(y_path),
    'model_path': str(lstm_model_path),
    'labels_path': str(labels_out_path),
    'input_shape': [int(X.shape[1]), int(X.shape[2])],
    'num_classes': int(num_classes),
    'validation_loss': float(val_loss),
    'validation_accuracy': float(val_acc),
    'epochs_ran': int(len(history_dict.get('loss', []))),
    'history': history_dict,
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Saved LSTM model:', lstm_model_path)
print('Saved class labels:', labels_out_path)
print('Saved metadata:', metadata_path)

In [ ]:
required = ['X', 'num_classes', 'val_acc', 'lstm_model_path', 'metadata_path']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing required objects before summary: {missing}')

summary = {
    'samples': int(X.shape[0]),
    'sequence_length': int(X.shape[1]),
    'feature_dim': int(X.shape[2]),
    'num_classes': int(num_classes),
    'val_accuracy': float(val_acc),
    'model_path': str(lstm_model_path),
    'metadata_path': str(metadata_path),
}
summary

In [ ]:
required = ['model', 'X_val', 'y_val', 'num_classes']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing required objects before evaluation: {missing}')

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
y_val_pred_probs = model.predict(X_val, verbose=0)
y_val_pred = np.argmax(y_val_pred_probs, axis=1)
y_val_true = np.argmax(y_val, axis=1)

print('Validation loss:', float(val_loss))
print('Validation accuracy:', float(val_acc))

cm = confusion_matrix(y_val_true, y_val_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, cmap='Blues', cbar=True)
plt.title('Confusion Matrix (Validation)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

if class_names is not None and len(class_names) == num_classes:
    target_names = [str(x) for x in class_names]
else:
    target_names = [f'class_{i}' for i in range(num_classes)]

report = classification_report(
    y_val_true,
    y_val_pred,
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)

print('Macro avg metrics:')
print({k: round(v, 4) for k, v in report['macro avg'].items()})

## 9) Evaluate Model Performance

In [ ]:
required = ['model', 'X_train', 'y_train', 'X_val', 'y_val', 'early_stopping']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing required objects before training: {missing}')

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1,
)

print('Epochs run:', len(history.history['loss']))

In [ ]:
model_obj = globals().get('model')
if model_obj is None:
    raise RuntimeError('Model is not defined. Run the model architecture cell first.')

model_obj.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
)

model = model_obj
print('Configured optimizer/loss/metrics and early stopping callback.')

## 7) Configure Loss, Optimizer, and Metrics

In [ ]:
if 'num_classes' not in globals():
    num_classes = int(len(np.unique(y)))

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.3),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax'),
])

model.summary()

In [14]:
# Optional troubleshooting cell for path resolution.
# Run this only if Cell 5 still cannot find data files.

import os
from pathlib import Path

print('cwd:', Path.cwd())
print('HAND_SIGN_REPO_ROOT:', os.getenv('HAND_SIGN_REPO_ROOT'))

roots_to_check = [
    Path.cwd(),
    Path('/content'),
    Path('/content/drive/MyDrive'),
    Path.home() / 'Documents' / 'hand_sign_detection_dynamic',
    Path('c:/Users/suman/Documents/hand_sign_detection_dynamic'),
    Path('/mnt/c/Users/suman/Documents/hand_sign_detection_dynamic'),
    Path('/app'),
]

for root in roots_to_check:
    has_data = (root / 'data' / 'X_data.npy').exists()
    print('candidate', root, 'exists', root.exists(), 'has_data', has_data)

if Path('/content/drive/MyDrive').exists():
    print('\nDrive-level candidates containing X_data.npy:')
    hits = 0
    try:
        for hit in Path('/content/drive/MyDrive').rglob('X_data.npy'):
            print(' -', hit)
            hits += 1
            if hits >= 10:
                print(' ... truncated after 10 hits')
                break
    except Exception as exc:
        print('Drive scan failed:', exc)

cwd: /content
HAND_SIGN_REPO_ROOT: None
candidate /content exists True has_data False
candidate /content exists True has_data False
candidate /content/drive/MyDrive exists True has_data False
candidate /root/Documents/hand_sign_detection_dynamic exists False has_data False
candidate c:/Users/suman/Documents/hand_sign_detection_dynamic exists False has_data False
candidate /mnt/c/Users/suman/Documents/hand_sign_detection_dynamic exists False has_data False
candidate /app exists False has_data False

Drive-level candidates containing X_data.npy:
